In [1]:
%pip install langchain_anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 833.9 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 5.8 MB/s eta 0:00:0000:0100:01


In [2]:
pip install -U langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 5.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.0
    Uninstalling langchain-core-1.4.0:
      Successfully uninstalled langchain-core-1.4.0


In [3]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Set your OpenAI API key (or load from .env)
# os.environ["OPENAI_API_KEY"] = "your-key-here"

print("LangChain imports loaded successfully")


LangChain imports loaded successfully


In [4]:
import os
from openai import OpenAI

In [24]:
# Initialize the model pointing to OpenRouter
llm = ChatOpenAI(
    model="openrouter/auto",                     # Any model available on OpenRouter
    api_key="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",            # Your OpenRouter API key
    base_url="https://openrouter.ai/api/v1",           # OpenRouter target endpoint
    temperature=0.7
)

In [6]:
import os
from langchain_openai import ChatOpenAI

# Set environment configurations
os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

# Initialize with environment defaults
llm = ChatOpenAI(model="google/gemini-2.5-pro")

response = llm.invoke("Hello!")
print(response.content)


Hello there! How can I help you today?


In [25]:
# Test the connection
response = llm.invoke("What is the distance between the Earth and the Moon?")
print(response.content)

The average distance between the Earth and the Moon is about **384,400 kilometers** (238,900 miles).

However, this distance varies slightly because the Moon's orbit is elliptical:
- **Closest point (perigee)**: ~356,500 km (221,500 miles)
- **Farthest point (apogee)**: ~406,700 km (252,700 miles)


In [26]:
response = llm.invoke("What is LangChain in one sentence?")
print(type(response))
print(response.content)


<class 'langchain_core.messages.ai.AIMessage'>
LangChain is a framework for developing applications powered by large language models (LLMs) by providing tools to connect them with external data sources, memory, and other components.


How Do Prompt Templates Help You Reuse Prompts?

In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains {topic} concepts simply."),
    ("human", "{question}")
])

formatted = prompt.invoke({
    "topic": "machine learning",
    "question": "What is overfitting?"
})
print(formatted.to_string())


System: You are a helpful assistant that explains machine learning concepts simply.
Human: What is overfitting?


In [10]:
simple_prompt = ChatPromptTemplate.from_template(
    "Translate '{text}' to {language}."
)
result = simple_prompt.invoke({"text": "Hello world", "language": "French"})
print(result.to_string())


Human: Translate 'Hello world' to French.


How Do Output Parsers Turn LLM Text into Structured Data?

In [11]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

raw_response = llm.invoke("Say hello")
print(f"Without parser: {type(raw_response).__name__}")

parsed = parser.invoke(raw_response)
print(f"With parser:    {type(parsed).__name__} -> {parsed}")


Without parser: AIMessage
With parser:    TextAccessor -> Hello there! How can I help you today?


JsonOutputParser — How Do You Get Python Dicts?

In [12]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the requested information. {format_instructions}"),
    ("human", "Give me the name, population, and continent of Japan.")
])

chain = prompt.partial(
    format_instructions=json_parser.get_format_instructions()
) | llm | json_parser

result = chain.invoke({})
print(type(result))
print(result)


<class 'dict'>
{'name': 'Japan', 'population': 125100000, 'continent': 'Asia'}


In [13]:
#PydanticOutputParser — How Do You Get Validated Output?
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class CountryInfo(BaseModel):
    name: str = Field(description="Name of the country")
    population: int = Field(description="Approximate population")
    continent: str = Field(description="Continent the country is in")

pydantic_parser = PydanticOutputParser(pydantic_object=CountryInfo)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract country information. {format_instructions}"),
    ("human", "Tell me about {country}.")
])

chain = prompt.partial(
    format_instructions=pydantic_parser.get_format_instructions()
) | llm | pydantic_parser

result = chain.invoke({"country": "Brazil"})
print(type(result))
print(f"Name: {result.name}")
print(f"Population: {result.population}")
print(f"Continent: {result.continent}")


<class '__main__.CountryInfo'>
Name: Brazil
Population: 215000000
Continent: South America


In [14]:
#How Do Chains Connect Everything with LCEL?
prompt = ChatPromptTemplate.from_template(
    "Explain {concept} in 2 sentences for a beginner."
)
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"concept": "neural networks"})
print(result)


A neural network is a computer system, inspired by the brain, that learns to recognize patterns by analyzing many examples. By adjusting its internal connections as it learns, it can then make intelligent predictions or decisions about new things it has never seen before.


In [15]:
#What Happens Under the Hood?
from langchain_core.runnables import RunnableSequence

chain = prompt | llm | StrOutputParser()

print(f"Chain type: {type(chain).__name__}")
print(f"Is RunnableSequence: {isinstance(chain, RunnableSequence)}")
print(f"Steps: {len(chain.steps)}")
for i, step in enumerate(chain.steps):
    print(f"  Step {i}: {type(step).__name__}")


Chain type: RunnableSequence
Is RunnableSequence: True
Steps: 3
  Step 0: ChatPromptTemplate
  Step 1: ChatOpenAI
  Step 2: StrOutputParser


In [16]:
#What Other Ways Can You Run a Chain?
results = chain.batch([
    {"concept": "API"},
    {"concept": "REST"},
])
for r in results:
    print(r[:80] + "...")
    print()


An API is like a waiter in a restaurant who acts as a messenger between you and ...

REST is a popular set of design rules for allowing different software applicatio...



In [17]:
for chunk in chain.stream({"concept": "machine learning"}):
    print(chunk, end="", flush=True)
print()


Machine learning teaches computers to find patterns in data on their own. They then use those learned patterns to make predictions or decisions about new things, improving with experience instead of being explicitly programmed for every task.


In [18]:
#How Do You Build a Real-World Chain?
from typing import List

class ActionItem(BaseModel):
    task: str = Field(description="Description of the action item")
    assignee: str = Field(description="Person responsible")

class MeetingSummary(BaseModel):
    title: str = Field(description="Brief meeting title")
    key_decisions: List[str] = Field(description="Main decisions made")
    action_items: List[ActionItem] = Field(description="Tasks assigned")


In [19]:
summary_parser = PydanticOutputParser(pydantic_object=MeetingSummary)

summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a meeting notes assistant. Extract structured "
     "information from the meeting transcript.\n"
     "{format_instructions}"),
    ("human", "{transcript}")
])

summary_chain = summary_prompt.partial(
    format_instructions=summary_parser.get_format_instructions()
) | llm | summary_parser


In [20]:
meeting_notes = """
Team standup March 10. Present: Alice, Bob, Carol.
Alice said the data pipeline is done and ready for review.
Bob mentioned the API rate limits are causing issues in production.
We decided to implement exponential backoff for API calls.
Carol will write the retry logic by Friday.
Bob will set up monitoring dashboards by next Tuesday.
Alice will review Carol's PR once it's ready.
"""

summary = summary_chain.invoke({"transcript": meeting_notes})
print(f"Title: {summary.title}")
print(f"\nKey Decisions:")
for decision in summary.key_decisions:
    print(f"  - {decision}")
print(f"\nAction Items:")
for item in summary.action_items:
    print(f"  - {item.task} (Assigned to: {item.assignee})")


Title: Team standup March 10

Key Decisions:
  - Implement exponential backoff for API calls.

Action Items:
  - Write the retry logic by Friday. (Assigned to: Carol)
  - Set up monitoring dashboards by next Tuesday. (Assigned to: Bob)
  - Review Carol's PR once it's ready. (Assigned to: Alice)


Exercise 1: Build a Translation Chain

In [21]:
ExerciseBlock:
  type: 'exercise'
  id: 'langchain-translation-chain'
  title: 'Build a Translation Chain'
  difficulty: 'beginner'
  exerciseType: 'write'
  instructions: |
    Create a LangChain chain that translates text between languages.
    1. Create a ChatPromptTemplate with variables {text} and {target_language}
    2. Chain it with the llm and StrOutputParser using the pipe operator
    3. Invoke the chain to translate "Good morning" to Spanish
    4. Print the result
  starterCode: |
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    # Step 1: Create the prompt template
    translate_prompt = ChatPromptTemplate.from_template(
        # Your template here -- include {text} and {target_language}
    )

    # Step 2: Build the chain with |
    translate_chain = # prompt | llm | parser

    # Step 3: Invoke and print
    result = translate_chain.invoke({
        "text": "Good morning",
        "target_language": "Spanish"
    })
    print(result)
  testCases:
    - id: 'tc1'
      input: 'print("DONE")'
      expectedOutput: 'DONE'
      description: 'Chain executes without errors'
    - id: 'tc2'
      input: 'print(type(result).__name__)'
      expectedOutput: 'str'
      description: 'Output should be a string'
  hints:
    - 'Template: "Translate the following text to {target_language}: {text}"'
    - 'Full chain: translate_prompt | llm | StrOutputParser()'
  solution: |
    translate_prompt = ChatPromptTemplate.from_template(
        "Translate the following text to {target_language}: {text}"
    )
    translate_chain = translate_prompt | llm | StrOutputParser()
    result = translate_chain.invoke({
        "text": "Good morning",
        "target_language": "Spanish"
    })
    print(result)
  solutionExplanation: |
    The prompt template uses {text} for the input and {target_language} for the target. The pipe operator chains template to model to parser. StrOutputParser extracts the plain text from the AIMessage.
  xpReward: 15


SyntaxError: invalid syntax (2117353565.py, line 1)

Exercise 2: Extract Structured Data with PydanticOutputParser

In [ ]:
ExerciseBlock:
  type: 'exercise'
  id: 'langchain-pydantic-parser'
  title: 'Extract Structured Data with PydanticOutputParser'
  difficulty: 'beginner'
  exerciseType: 'write'
  instructions: |
    Create a chain that extracts book information into a Pydantic model.
    1. Complete the BookInfo model by adding author (str) and year (int) fields
    2. The parser and prompt are already set up
    3. Build the chain and invoke it with the given text
    4. Print the title, author, and year
  starterCode: |
    from pydantic import BaseModel, Field
    from langchain_core.output_parsers import PydanticOutputParser
    from langchain_core.prompts import ChatPromptTemplate

    # Step 1: Complete the Pydantic model
    class BookInfo(BaseModel):
        title: str = Field(description="Title of the book")
        # Add author and year fields here

    # Step 2: Parser and prompt (already done)
    book_parser = PydanticOutputParser(pydantic_object=BookInfo)
    book_prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract book information. {format_instructions}"),
        ("human", "{text}")
    ])

    # Step 3: Build and invoke the chain
    book_chain = book_prompt.partial(
        format_instructions=book_parser.get_format_instructions()
    ) | llm | book_parser

    result = book_chain.invoke({
        "text": "The Great Gatsby was written by F. Scott Fitzgerald in 1925"
    })
    print(f"Title: {result.title}")
    print(f"Author: {result.author}")
    print(f"Year: {result.year}")
  testCases:
    - id: 'tc1'
      input: 'print(type(result).__name__)'
      expectedOutput: 'BookInfo'
      description: 'Result should be a BookInfo object'
    - id: 'tc2'
      input: 'print(result.year)'
      expectedOutput: '1925'
      description: 'Year should be 1925'
    - id: 'tc3'
      input: 'print("DONE")'
      expectedOutput: 'DONE'
      description: 'Code runs without errors'
      hidden: true
  hints:
    - 'Add: author: str = Field(description="Author of the book")'
    - 'Add: year: int = Field(description="Publication year")'
  solution: |
    class BookInfo(BaseModel):
        title: str = Field(description="Title of the book")
        author: str = Field(description="Author of the book")
        year: int = Field(description="Publication year")

    book_parser = PydanticOutputParser(pydantic_object=BookInfo)
    book_prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract book information. {format_instructions}"),
        ("human", "{text}")
    ])
    book_chain = book_prompt.partial(
        format_instructions=book_parser.get_format_instructions()
    ) | llm | book_parser
    result = book_chain.invoke({
        "text": "The Great Gatsby was written by F. Scott Fitzgerald in 1925"
    })
    print(f"Title: {result.title}")
    print(f"Author: {result.author}")
    print(f"Year: {result.year}")
  solutionExplanation: |
    The Pydantic model defines typed fields for structured extraction. PydanticOutputParser generates format instructions telling the LLM what JSON shape to produce. The parser then validates and converts the response into a BookInfo object with proper type coercion.
  xpReward: 20
